# 04 — Trajectory Evaluation (baseline vs calibration results)

Generates the **ground truth** elliptical trajectory (2 intersecting ellipses, 45° and 135°
rotation, forward + backward segments), then compares it with the trajectory produced by the
virtual encoder using **nearest neighbour** (`scipy.cKDTree`). RMSE is expressed in **microns**.

**Inputs:**
- `data/raw/circular_baseline/hasil_fix_banget_baseline_XX.csv`
- `data/raw/circular_calibrated/JALUR_VIDEO_FINAL_3_XX.csv`

**Outputs:** evaluation plots to `figures/`, plus an RMSE table and the accuracy improvement
percentage printed on screen.

In [ ]:
# ==========================================================================
# LOAD PATH CONFIGURATION  (single source for all notebooks: config_paths.py)
# ==========================================================================
import os
import sys

_p = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(_p, "config_paths.py")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError(
            "config_paths.py not found. Run this notebook from inside the "
            "organized repo folder, or set the KALIBRASI_REPO environment variable to the repo root."
        )
    _p = _parent
sys.path.insert(0, _p)
import config_paths as CFG

print("Ground truth :", CFG.ELLIPSE)
print("CSV baseline :", CFG.CSV_DIR_BASELINE)
print("CSV calibrated:", CFG.CSV_DIR_CALIBRATED)
print("Output images :", CFG.FIG_DIR)
print()


In [ ]:
import math
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree


# --- 1. Ground Truth Functions ---
def get_ellipse_point(center_x, center_y, a, b, angle_deg, t):
    """Computes absolute X, Y coordinates on an ellipse with a given rotation."""
    angle_rad = math.radians(angle_deg)
    x = a * math.cos(t)
    y = b * math.sin(t)
    x_rot = x * math.cos(angle_rad) - y * math.sin(angle_rad)
    y_rot = x * math.sin(angle_rad) + y * math.cos(angle_rad)
    return center_x + x_rot, center_y + y_rot


def generate_path_pts(
    center_x, center_y, a, b, angle_deg, t_start, t_end, num_points
):
    """Generates an array of point coordinates from start angle to end angle."""
    pts = []
    step = (t_end - t_start) / num_points
    for i in range(0, num_points + 1):
        t = t_start + i * step
        x, y = get_ellipse_point(center_x, center_y, a, b, angle_deg, t)
        pts.append([x, y])
    return pts


# --- 2. Ground Truth Configuration ---
center_x = 0
center_y = -3.162277
skala = 10.0
ratio = 8
a = 4.00 * skala / ratio  # Semi-major axis = 5.0 mm
b = 2.00 * skala / ratio  # Semi-minor axis = 2.5 mm
resolusi_titik = 240

# Angle intersection points
t_start_1 = math.atan2(a, b)
x0, y0 = get_ellipse_point(center_x, center_y, a, b, 45, t_start_1)
print(f"Initial GT Point (Before Mirror): ({x0:.4f}, {y0:.4f})\n")

t_start_2 = math.atan2(-a, b)

gt_pts = []
# Forward Motion
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        45,
        t_start_1,
        t_start_1 + 2 * math.pi,
        resolusi_titik,
    )
)
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        135,
        t_start_2,
        t_start_2 + 2 * math.pi,
        resolusi_titik,
    )
)
# Backward Motion
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        135,
        t_start_2 + 2 * math.pi,
        t_start_2,
        resolusi_titik,
    )
)
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        45,
        t_start_1 + 2 * math.pi,
        t_start_1,
        resolusi_titik,
    )
)

gt_pts = np.array(gt_pts)

# --- CHANGE: Mirror the Y axis on the Ground Truth ---
gt_pts[:, 1] = -gt_pts[:, 1]

# Build cKDTree ONLY ONCE outside the loop to speed up computation
kdtree = cKDTree(gt_pts)

# --- 3. Batch Processing Setup ---
# Manual File Index List
file_indices = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10"]

# Result storage variables for computing the mean and standard deviation
all_rmse_x = []
all_rmse_y = []
all_rmse_total = []

# --- 4. Evaluation Process Loop ---
for idx in file_indices:
    filename_csv = os.path.join(CFG.CSV_DIR_BASELINE, CFG.CSV_BASELINE_NAME.format(int(idx)))
    output_plot_name = os.path.join(CFG.FIG_DIR, f"baseline_evaluation_{idx}.png")

    # Check that the file exists before processing
    if not os.path.exists(filename_csv):
        print(f"File {filename_csv} not found, skipping...\n")
        continue

    print(f"Processing {filename_csv} ...")

    # Load data
    df = pd.read_csv(filename_csv)

    # Original CSV data WITHOUT any mirroring
    csv_x = df["X Posisi (mm)"].values
    csv_y = df["Y Posisi (mm)"].values
    csv_pts = np.column_stack((csv_x, csv_y))

    # Translation so the CSV start point aligns with the GT start point
    csv_pts = csv_pts - csv_pts[0] + gt_pts[0]
    csv_x = csv_pts[:, 0]
    csv_y = csv_pts[:, 1]

    # Compute RMSE using the Nearest Neighbor algorithm
    distances, indices = kdtree.query(csv_pts)
    closest_pts = gt_pts[indices]

    # Convert to microns (* 1000)
    rmse_x = np.sqrt(np.mean((csv_pts[:, 0] - closest_pts[:, 0]) ** 2)) * 1000
    rmse_y = np.sqrt(np.mean((csv_pts[:, 1] - closest_pts[:, 1]) ** 2)) * 1000
    rmse_total = np.sqrt(np.mean(distances**2)) * 1000

    # Store RMSE values in the lists for the final computation
    all_rmse_x.append(rmse_x)
    all_rmse_y.append(rmse_y)
    all_rmse_total.append(rmse_total)

    print(f"  RMSE X      : {rmse_x:.2f} µm")
    print(f"  RMSE Y      : {rmse_y:.2f} µm")
    print(f"  RMSE Total  : {rmse_total:.2f} µm\n")

    # --- 5. Per-File Plot Visualization ---
    plt.figure(figsize=(10, 8))
    plt.plot(
        gt_pts[:, 0],
        gt_pts[:, 1],
        label="Ground Truth",
        color="black",
        linestyle="--",
        linewidth=2,
    )
    plt.plot(
        csv_x, csv_y, label=f"Virtual Encoder ({idx})", color="red", alpha=0.7
    )

    plt.title(f"Machine Baseline Performance Evaluation_{idx}")
    plt.xlabel("X Position (mm)")
    plt.ylabel("Y Position (mm)")
    plt.axis("equal")
    plt.grid(True)
    plt.legend()

    # Add a text box for the metrics in microns
    text_str = f"RMSE X: {rmse_x:.2f} µm\nRMSE Y: {rmse_y:.2f} µm\nRMSE Total: {rmse_total:.2f} µm"
    plt.text(
        0.05,
        0.95,
        text_str,
        transform=plt.gca().transAxes,
        fontsize=12,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    )

    # Save the plot automatically
    plt.savefig(output_plot_name, dpi=300, bbox_inches="tight")
    plt.close()

# --- 6. Overall Results Summary ---
print("All processes completed! The evaluation plots have been saved.")

if all_rmse_total:
    avg_rmse_x = np.mean(all_rmse_x)
    std_rmse_x = np.std(all_rmse_x)

    avg_rmse_y = np.mean(all_rmse_y)
    std_rmse_y = np.std(all_rmse_y)

    avg_rmse_total = np.mean(all_rmse_total)
    std_rmse_total = np.std(all_rmse_total)

    print("\n" + "=" * 50)
    print("OVERALL EVALUATION RESULTS SUMMARY (Unit: Microns)")
    print("=" * 50)
    print(
        f"Mean RMSE X          : {avg_rmse_x:.2f} µm (Std Dev: {std_rmse_x:.2f} µm)"
    )
    print(
        f"Mean RMSE Y          : {avg_rmse_y:.2f} µm (Std Dev: {std_rmse_y:.2f} µm)"
    )
    print(
        f"Mean RMSE Total      : {avg_rmse_total:.2f} µm (Std Dev: {std_rmse_total:.2f} µm)"
    )
    print("=" * 50)

In [ ]:
import math
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree


# --- 1. Ground Truth Functions ---
def get_ellipse_point(center_x, center_y, a, b, angle_deg, t):
    """Computes absolute X, Y coordinates on an ellipse with a given rotation."""
    angle_rad = math.radians(angle_deg)
    x = a * math.cos(t)
    y = b * math.sin(t)
    x_rot = x * math.cos(angle_rad) - y * math.sin(angle_rad)
    y_rot = x * math.sin(angle_rad) + y * math.cos(angle_rad)
    return center_x + x_rot, center_y + y_rot


def generate_path_pts(
    center_x, center_y, a, b, angle_deg, t_start, t_end, num_points
):
    """Generates an array of point coordinates from start angle to end angle."""
    pts = []
    step = (t_end - t_start) / num_points
    for i in range(0, num_points + 1):
        t = t_start + i * step
        x, y = get_ellipse_point(center_x, center_y, a, b, angle_deg, t)
        pts.append([x, y])
    return pts


# --- 2. Ground Truth Configuration ---
center_x = 0
center_y = -3.162277
skala = 10.0
ratio = 8
a = 4.00 * skala / ratio  # Semi-major axis = 5.0 mm
b = 2.00 * skala / ratio  # Semi-minor axis = 2.5 mm
resolusi_titik = 240

# Angle intersection points
t_start_1 = math.atan2(a, b)
x0, y0 = get_ellipse_point(center_x, center_y, a, b, 45, t_start_1)
print(f"Initial GT Point (Before Mirror): ({x0:.4f}, {y0:.4f})\n")

t_start_2 = math.atan2(-a, b)

gt_pts = []
# Forward Motion
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        45,
        t_start_1,
        t_start_1 + 2 * math.pi,
        resolusi_titik,
    )
)
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        135,
        t_start_2,
        t_start_2 + 2 * math.pi,
        resolusi_titik,
    )
)
# Backward Motion
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        135,
        t_start_2 + 2 * math.pi,
        t_start_2,
        resolusi_titik,
    )
)
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        45,
        t_start_1 + 2 * math.pi,
        t_start_1,
        resolusi_titik,
    )
)

gt_pts = np.array(gt_pts)

# --- CHANGE: Mirror the Y axis on the Ground Truth ---
gt_pts[:, 1] = -gt_pts[:, 1]

# Build cKDTree ONLY ONCE outside the loop to speed up computation
kdtree = cKDTree(gt_pts)

# --- 3. Batch Processing Setup ---
# Manual File Index List
file_indices = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10"]

# Result storage variables for computing the mean and standard deviation
all_rmse_x = []
all_rmse_y = []
all_rmse_total = []

# --- 4. Evaluation Process Loop ---
for idx in file_indices:
    filename_csv = os.path.join(CFG.CSV_DIR_CALIBRATED, CFG.CSV_CALIBRATED_NAME.format(int(idx)))
    output_plot_name = os.path.join(CFG.FIG_DIR, f"calibrated_evaluation_{idx}.png")

    # Check that the file exists before processing
    if not os.path.exists(filename_csv):
        print(f"File {filename_csv} not found, skipping...\n")
        continue

    print(f"Processing {filename_csv} ...")

    # Load data
    df = pd.read_csv(filename_csv)

    # Original CSV data WITHOUT any mirroring
    csv_x = df["X Posisi (mm)"].values
    csv_y = df["Y Posisi (mm)"].values
    csv_pts = np.column_stack((csv_x, csv_y))

    # Translation so the CSV start point aligns with the GT start point
    csv_pts = csv_pts - csv_pts[0] + gt_pts[0]
    csv_x = csv_pts[:, 0]
    csv_y = csv_pts[:, 1]

    # Compute RMSE using the Nearest Neighbor algorithm
    distances, indices = kdtree.query(csv_pts)
    closest_pts = gt_pts[indices]

    # Convert to microns (* 1000)
    rmse_x = np.sqrt(np.mean((csv_pts[:, 0] - closest_pts[:, 0]) ** 2)) * 1000
    rmse_y = np.sqrt(np.mean((csv_pts[:, 1] - closest_pts[:, 1]) ** 2)) * 1000
    rmse_total = np.sqrt(np.mean(distances**2)) * 1000

    # Store RMSE values in the lists for the final computation
    all_rmse_x.append(rmse_x)
    all_rmse_y.append(rmse_y)
    all_rmse_total.append(rmse_total)

    print(f"  RMSE X      : {rmse_x:.2f} µm")
    print(f"  RMSE Y      : {rmse_y:.2f} µm")
    print(f"  RMSE Total  : {rmse_total:.2f} µm\n")

    # --- 5. Per-File Plot Visualization ---
    plt.figure(figsize=(10, 8))
    plt.plot(
        gt_pts[:, 0],
        gt_pts[:, 1],
        label="Ground Truth",
        color="black",
        linestyle="--",
        linewidth=2,
    )
    plt.plot(
        csv_x, csv_y, label=f"Virtual Encoder ({idx})", color="red", alpha=0.7
    )

    plt.title(f"Machine Calibration Result Performance Evaluation_{idx}")
    plt.xlabel("X Position (mm)")
    plt.ylabel("Y Position (mm)")
    plt.axis("equal")
    plt.grid(True)
    plt.legend()

    # Add a text box for the metrics in microns
    text_str = f"RMSE X: {rmse_x:.2f} µm\nRMSE Y: {rmse_y:.2f} µm\nRMSE Total: {rmse_total:.2f} µm"
    plt.text(
        0.05,
        0.95,
        text_str,
        transform=plt.gca().transAxes,
        fontsize=12,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    )

    # Save the plot automatically
    plt.savefig(output_plot_name, dpi=300, bbox_inches="tight")
    plt.close()

# --- 6. Overall Results Summary ---
print("All processes completed! The evaluation plots have been saved.")

if all_rmse_total:
    avg_rmse_x = np.mean(all_rmse_x)
    std_rmse_x = np.std(all_rmse_x)

    avg_rmse_y = np.mean(all_rmse_y)
    std_rmse_y = np.std(all_rmse_y)

    avg_rmse_total = np.mean(all_rmse_total)
    std_rmse_total = np.std(all_rmse_total)

    print("\n" + "=" * 50)
    print("OVERALL EVALUATION RESULTS SUMMARY (Unit: Microns)")
    print("=" * 50)
    print(
        f"Mean RMSE X          : {avg_rmse_x:.2f} µm (Std Dev: {std_rmse_x:.2f} µm)"
    )
    print(
        f"Mean RMSE Y          : {avg_rmse_y:.2f} µm (Std Dev: {std_rmse_y:.2f} µm)"
    )
    print(
        f"Mean RMSE Total      : {avg_rmse_total:.2f} µm (Std Dev: {std_rmse_total:.2f} µm)"
    )
    print("=" * 50)

In [ ]:
print(f"Initial CSV point (after mirror): ({csv_pts[0,0]:.4f}, {csv_pts[0,1]:.4f})")
print(f"Initial CSV raw point: ({df['X Posisi (mm)'].values[0]:.4f}, {df['Y Posisi (mm)'].values[0]:.4f})")


In [ ]:
print(f"=== CSV Range (after mirror) ===")
print(f"X: [{csv_pts[:,0].min():.4f}, {csv_pts[:,0].max():.4f}]")
print(f"Y: [{csv_pts[:,1].min():.4f}, {csv_pts[:,1].max():.4f}]")

print(f"\n=== Ground Truth Range ===")
print(f"X: [{gt_pts[:,0].min():.4f}, {gt_pts[:,0].max():.4f}]")
print(f"Y: [{gt_pts[:,1].min():.4f}, {gt_pts[:,1].max():.4f}]")

print(f"\n=== CSV Center (mean) ===")
print(f"X center: {csv_pts[:,0].mean():.4f}")
print(f"Y center: {csv_pts[:,1].mean():.4f}")

print(f"\n=== GT Center (mean) ===")
print(f"X center: {gt_pts[:,0].mean():.4f}")
print(f"Y center: {gt_pts[:,1].mean():.4f}")


In [ ]:
import math
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# ==========================================
# 1. IEEE GRAPH & FONT CONFIGURATION
# ==========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'Liberation Serif'],
    'font.size': 14,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'mathtext.fontset': 'stix',
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})


# --- 2. Ground Truth Functions ---
def get_ellipse_point(center_x, center_y, a, b, angle_deg, t):
    """Calculates absolute X, Y coordinates on an ellipse with a specific rotation."""
    angle_rad = math.radians(angle_deg)
    x = a * math.cos(t)
    y = b * math.sin(t)
    x_rot = x * math.cos(angle_rad) - y * math.sin(angle_rad)
    y_rot = x * math.sin(angle_rad) + y * math.cos(angle_rad)
    return center_x + x_rot, center_y + y_rot


def generate_path_pts(
    center_x, center_y, a, b, angle_deg, t_start, t_end, num_points
):
    """Generates an array of point coordinates from start angle to end angle."""
    pts = []
    step = (t_end - t_start) / num_points
    for i in range(0, num_points + 1):
        t = t_start + i * step
        x, y = get_ellipse_point(center_x, center_y, a, b, angle_deg, t)
        pts.append([x, y])
    return pts


# --- 3. Ground Truth Configuration ---
center_x = 0
center_y = -3.162277
skala = 10.0
ratio = 8
a = 4.00 * skala / ratio  # Semi-major axis = 5.0 mm
b = 2.00 * skala / ratio  # Semi-minor axis = 2.5 mm
resolusi_titik = 240

# Angle intersection points
t_start_1 = math.atan2(a, b)
x0, y0 = get_ellipse_point(center_x, center_y, a, b, 45, t_start_1)
print(f"Initial GT Point (Before Mirror): ({x0:.4f}, {y0:.4f})\n")

t_start_2 = math.atan2(-a, b)

gt_pts = []
# Forward Motion
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        45,
        t_start_1,
        t_start_1 + 2 * math.pi,
        resolusi_titik,
    )
)
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        135,
        t_start_2,
        t_start_2 + 2 * math.pi,
        resolusi_titik,
    )
)
# Backward Motion
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        135,
        t_start_2 + 2 * math.pi,
        t_start_2,
        resolusi_titik,
    )
)
gt_pts.extend(
    generate_path_pts(
        center_x,
        center_y,
        a,
        b,
        45,
        t_start_1 + 2 * math.pi,
        t_start_1,
        resolusi_titik,
    )
)

gt_pts = np.array(gt_pts)

# Mirror Y-axis for Ground Truth
gt_pts[:, 1] = -gt_pts[:, 1]

# Build cKDTree for nearest-neighbor search
kdtree = cKDTree(gt_pts)

# --- 4. Batch Processing Setup ---
file_indices = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10"]

# Storage for Baseline Performance results
base_rmse_x_list = []
base_rmse_y_list = []
base_rmse_total_list = []

# Storage for Calibrated Performance results
calib_rmse_x_list = []
calib_rmse_y_list = []
calib_rmse_total_list = []

# --- 5. Evaluation & Comparison Loop ---
for idx in file_indices:
    filename_csv_calib = os.path.join(CFG.CSV_DIR_CALIBRATED, CFG.CSV_CALIBRATED_NAME.format(int(idx)))
    filename_csv_base = os.path.join(CFG.CSV_DIR_BASELINE, CFG.CSV_BASELINE_NAME.format(int(idx)))
    output_plot_name = os.path.join(CFG.FIG_DIR, f"comparison_full_{idx}.png")

    # Check file existence
    if not os.path.exists(filename_csv_calib) or not os.path.exists(filename_csv_base):
        print(f"File index {idx} incomplete (Calib/Baseline missing), skipping...\n")
        continue

    print(f"Processing data index {idx}...")

    # --- Load & Process Calibrated Performance Data ---
    df_calib = pd.read_csv(filename_csv_calib)
    pts_calib = np.column_stack((df_calib["X Posisi (mm)"].values, df_calib["Y Posisi (mm)"].values))
    pts_calib = pts_calib - pts_calib[0] + gt_pts[0]  # Start-point alignment to GT

    distances_calib, indices_calib = kdtree.query(pts_calib)
    closest_calib = gt_pts[indices_calib]

    rmse_x_calib = np.sqrt(np.mean((pts_calib[:, 0] - closest_calib[:, 0]) ** 2)) * 1000
    rmse_y_calib = np.sqrt(np.mean((pts_calib[:, 1] - closest_calib[:, 1]) ** 2)) * 1000
    rmse_tot_calib = np.sqrt(np.mean(distances_calib**2)) * 1000

    calib_rmse_x_list.append(rmse_x_calib)
    calib_rmse_y_list.append(rmse_y_calib)
    calib_rmse_total_list.append(rmse_tot_calib)

    # --- Load & Process Baseline Performance Data ---
    df_base = pd.read_csv(filename_csv_base)
    pts_base = np.column_stack((df_base["X Posisi (mm)"].values, df_base["Y Posisi (mm)"].values))
    pts_base = pts_base - pts_base[0] + gt_pts[0]  # Start-point alignment to GT

    distances_base, indices_base = kdtree.query(pts_base)
    closest_base = gt_pts[indices_base]

    rmse_x_base = np.sqrt(np.mean((pts_base[:, 0] - closest_base[:, 0]) ** 2)) * 1000
    rmse_y_base = np.sqrt(np.mean((pts_base[:, 1] - closest_base[:, 1]) ** 2)) * 1000
    rmse_tot_base = np.sqrt(np.mean(distances_base**2)) * 1000

    base_rmse_x_list.append(rmse_x_base)
    base_rmse_y_list.append(rmse_y_base)
    base_rmse_total_list.append(rmse_tot_base)

    print(f"  [Baseline Performance]   RMSE Total: {rmse_tot_base:.2f} µm")
    print(f"  [Calibrated Performance] RMSE Total: {rmse_tot_calib:.2f} µm\n")

    # --- 6. Visualization & Plotting ---
    plt.figure(figsize=(10, 8))
    
    # 1. Ground Truth: Dashed Black
    plt.plot(
        gt_pts[:, 0],
        gt_pts[:, 1],
        label="Ground Truth",
        color="black",
        linestyle="--",
        linewidth=2.5,
        zorder=1,
    )
    
    # 2. Baseline Performance: Dotted Blue
    plt.plot(
        pts_base[:, 0],
        pts_base[:, 1],
        label=f"Baseline Performance ({idx})",
        color="#0055ff",
        linestyle=":",
        linewidth=3.0,
        alpha=0.9,
        zorder=2,
    )
    
    # 3. Calibrated Performance: Solid Red
    plt.plot(
        pts_calib[:, 0],
        pts_calib[:, 1],
        label=f"Calibrated Performance ({idx})",
        color="#d62728",
        linestyle="-",
        linewidth=2.0,
        alpha=0.85,
        zorder=3,
    )

    plt.title(f"Trajectory Performance Comparison - Trial {idx}", fontsize=14)
    plt.xlabel("X Position (mm)", fontsize=14)
    plt.ylabel("Y Position (mm)", fontsize=14)
    plt.axis("equal")
    plt.grid(True)
    plt.legend(loc="upper right")

    # Metric comparison text box - Centered box with left-aligned text
    text_str = (
        f"Baseline Performance\n"
        f"• RMSE X = {rmse_x_base:.2f} µm\n"
        f"• RMSE Y = {rmse_y_base:.2f} µm\n"
        f"• RMSE Total = {rmse_tot_base:.2f} µm\n\n"
        f"Calibrated Performance\n"
        f"• RMSE X = {rmse_x_calib:.2f} µm\n"
        f"• RMSE Y = {rmse_y_calib:.2f} µm\n"
        f"• RMSE Total = {rmse_tot_calib:.2f} µm"
    )
    
    plt.text(
        0.5,
        0.5,
        text_str,
        transform=plt.gca().transAxes,
        fontsize=12,
        horizontalalignment="center",
        verticalalignment="center",
        multialignment="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
    )

    plt.savefig(output_plot_name, dpi=300, bbox_inches="tight")
    plt.close()

# --- 7. Overall Summary ---
print("All comparative evaluation processes completed!")

if calib_rmse_total_list and base_rmse_total_list:
    print("\n" + "=" * 65)
    print("FULL COMPARATIVE EVALUATION SUMMARY (Unit: Microns)")
    print("=" * 65)
    print(
        f"Baseline Performance:\n"
        f"  Average RMSE X     : {np.mean(base_rmse_x_list):.2f} µm (Std: {np.std(base_rmse_x_list):.2f} µm)\n"
        f"  Average RMSE Y     : {np.mean(base_rmse_y_list):.2f} µm (Std: {np.std(base_rmse_y_list):.2f} µm)\n"
        f"  Average RMSE Total : {np.mean(base_rmse_total_list):.2f} µm (Std: {np.std(base_rmse_total_list):.2f} µm)\n"
    )
    print(
        f"Calibrated Performance:\n"
        f"  Average RMSE X     : {np.mean(calib_rmse_x_list):.2f} µm (Std: {np.std(calib_rmse_x_list):.2f} µm)\n"
        f"  Average RMSE Y     : {np.mean(calib_rmse_y_list):.2f} µm (Std: {np.std(calib_rmse_y_list):.2f} µm)\n"
        f"  Average RMSE Total : {np.mean(calib_rmse_total_list):.2f} µm (Std: {np.std(calib_rmse_total_list):.2f} µm)\n"
    )
    
    improvement_pct = ((np.mean(base_rmse_total_list) - np.mean(calib_rmse_total_list)) / np.mean(base_rmse_total_list)) * 100
    print(f"Total Accuracy Improvement: {improvement_pct:.2f}%")
    print("=" * 65)